In [ ]:
%matplotlib inline
import scanpy as sc
import anndata as ad

import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import os 
import warnings
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Preprocessing

Access Data

In [ ]:
basedir = '../../data/cellbender/paper_raw/'
samps = os.listdir(basedir)
filt_files = [os.path.join(basedir,s,'output_filtered.h5') for s in samps]

def get_cellbender_data(directory:str):
    data = sc.read_10x_h5(directory)
    identifier = directory.split('/')[-2]
    data.obs['Identifier'] = identifier
    data.obs['Experiment'],data.obs['Condition'],data.obs['Sample'] = identifier.split('_')
    data.obs.index = data.obs.index+'-'+data.obs['Sample']
    return data

data = [get_cellbender_data(file) for file in filt_files]

## QC

### Get QC Metrics

In [ ]:
def QC_Filter(data):
    sc.pp.filter_cells(data, min_genes=250)
    sc.pp.filter_cells(data, min_counts=500)
    sc.pp.filter_genes(data, min_cells=10)
    data.var_names_make_unique()
    
    data.var["mt"] = data.var_names.str.startswith("Mt")
    data.var["ribo"] = data.var_names.str.startswith(("Rps", "Rpl"))
    sc.pp.calculate_qc_metrics(data, qc_vars=["mt", "ribo"], inplace=True, percent_top=[20], log1p=True)

    remove = ['total_counts_mt', 'log1p_total_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo']
    data.obs = data.obs[[x for x in data.obs.columns if x not in remove]]
    return data

data = [QC_Filter(dat) for dat in data]

### QC Plots

In [ ]:
# sns.displot(data[1].obs['total_counts'], bins=100, kde=False).figure
tmp = data[1]
sc.pl.violin(
    tmp,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True
)
sc.pl.scatter(tmp, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

Plot QC Metrics for all datasets

In [ ]:
def QC_Plot(df, value):
    sns.set(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

    g = sns.FacetGrid(df, row="Sample", hue="Sample", aspect=15, height=0.5, palette="tab20")

    g.map(sns.kdeplot, value, clip_on=False, fill=True, alpha=1, linewidth=1.5)
    g.map(sns.kdeplot, value, clip_on=False, color="w", lw=2)

    g.map(plt.axhline, y=0, lw=2, clip_on=False)

    def label(x, color, label):
        ax = plt.gca()
        ax.text(0, .2, label, fontweight="bold", color=color,
                ha="left", va="center", transform=ax.transAxes)

    g.map(label, value)

    g.figure.subplots_adjust(hspace=-.6)

    g.set_titles("")
    g.set(yticks=[], ylabel="")
    g.despine(bottom=True, left=True)

    for ax in g.axes.flat:
        ax.axvline(x=df[value].median(), color='r', linestyle='-')

    return g.figure

df = pd.concat([x.obs for x in data])
df = df.sort_values('Sample')
QC_Plot(df,"log1p_total_counts")
# QC_Plot(df,"pct_counts_mt")
# QC_Plot(df,"n_genes")
# QC_Plot(df,"pct_counts_in_top_20_genes")

### Doublets

#### Option 1: scDblFinder
See: https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html#doublet-detection

In [4]:
import rpy2.ipython
%load_ext rpy2.ipython

In [ ]:
%reload_ext rpy2.ipython
%%R
install.packages(c("Seurat", "scater", "scDblFinder", "BiocParallel"))
library(Seurat)
library(scater)
library(scDblFinder)
library(BiocParallel)

data_mat = tmp.X.T

--- Please select a CRAN mirror for use in this session ---


Secure CRAN mirrors 

 1: 0-Cloud [https]
 2: Australia (Canberra) [https]
 3: Australia (Melbourne 1) [https]
 4: Australia (Melbourne 2) [https]
 5: Austria (Wien 1) [https]
 6: Belgium (Brussels) [https]
 7: Brazil (PR) [https]
 8: Brazil (SP 1) [https]
 9: Brazil (SP 2) [https]
10: Bulgaria [https]
11: Canada (MB) [https]
12: Canada (ON 1) [https]
13: Canada (ON 2) [https]
14: Chile (Santiago) [https]
15: China (Beijing 1) [https]
16: China (Beijing 2) [https]
17: China (Beijing 3) [https]
18: China (Hefei) [https]
19: China (Hong Kong) [https]
20: China (Jinan) [https]
21: China (Lanzhou) [https]
22: China (Nanjing) [https]
23: China (Shanghai 2) [https]
24: China (Shenzhen) [https]
25: China (Wuhan) [https]
26: Colombia (Cali) [https]
27: Cyprus [https]
28: Czech Republic [https]
29: Denmark [https]
30: East Asia [https]
31: Ecuador (Cuenca) [https]
32: Finland (Helsinki) [https]
33: France (Lyon 1) [https]
34: France (Lyon 2) [https]
35: France (Paris 1) [https]
36: Germany (Erl

R[write to console]: Warning:
R[write to console]:  dependency ‘Matrix’ is not available

R[write to console]: also installing the dependencies ‘bitops’, ‘dotCall64’, ‘rprojroot’, ‘gtools’, ‘caTools’, ‘Rcpp’, ‘tensor’, ‘BH’, ‘sitmo’, ‘sp’, ‘spam’, ‘cpp11’, ‘crosstalk’, ‘RcppTOML’, ‘here’, ‘gplots’, ‘gridExtra’, ‘RcppArmadillo’, ‘spatstat.data’, ‘spatstat.univar’, ‘spatstat.random’, ‘spatstat.utils’, ‘spatstat.sparse’, ‘goftest’, ‘abind’, ‘deldir’, ‘polyclip’, ‘FNN’, ‘dqrng’, ‘SeuratObject’, ‘cowplot’, ‘fastDummies’, ‘fitdistrplus’, ‘ggrepel’, ‘ggridges’, ‘ica’, ‘igraph’, ‘irlba’, ‘leidenbase’, ‘lmtest’, ‘matrixStats’, ‘miniUI’, ‘patchwork’, ‘pbapply’, ‘plotly’, ‘png’, ‘RANN’, ‘RcppAnnoy’, ‘RcppHNSW’, ‘reticulate’, ‘ROCR’, ‘RSpectra’, ‘Rtsne’, ‘scattermore’, ‘sctransform’, ‘spatstat.explore’, ‘spatstat.geom’, ‘uwot’, ‘RcppProgress’


R[write to console]: trying URL 'https://lib.stat.cmu.edu/R/CRAN/src/contrib/bitops_1.0-9.tar.gz'

R[write to console]: Content type 'application/x-gzip'
R

x86_64-conda-linux-gnu-cc -I"/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -Wl,-rpath-link,/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib    -fpic  -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -fdebug-prefix-map=/croot/r-base_1744617423037/work=/usr/local/src/conda/r-base-4.3.1 -fdebug-prefix-map=/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq=/usr/local/src/conda-prefix  -c bit-ops.c -o bit-ops.o
x86_64-conda-linux-gnu-cc -I"/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -Wl,-rpath-link,/mnt/DAT

installing to /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/library/00LOCK-bitops/00new/bitops/libs
** R
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (bitops)
* installing *source* package ‘dotCall64’ ...
** package ‘dotCall64’ successfully unpacked and MD5 sums checked
** using staged installation
** libs
using C compiler: ‘x86_64-conda-linux-gnu-cc (Anaconda gcc) 11.2.0’
using Fortran compiler: ‘GNU Fortran (Anaconda gcc) 11.2.0’


...

x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/include" -DNDEBUG  -I'/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/library/Rcpp/include' -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -Wl,-rpath-link,/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib    -fpic  -fvisibility-inlines-hidden  -fmessage-length=0 -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -fdebug-prefix-map=/croot/r-base_1744617423037/work=/usr/local/src/conda/r-base-4.3.1 -fdebug-prefix-map=/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq=/usr/local/src/conda-prefix  -c RcppExports.cpp -o RcppExports.o
x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/include" -DNDEBUG  -I'/mnt/DATA/hom

installing to /mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/library/00LOCK-ggrepel/00new/ggrepel/libs
** R
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (ggrepel)
* installing *source* package ‘igraph’ ...
** package ‘igraph’ successfully unpacked and MD5 sums checked
** using staged installation


libxml2 include directories: -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include/libxml2 -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include
libxml2 library link flags: -L/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib -lxml2 -L/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib -lz -L/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib -llzma -L/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib -licui18n -licuuc -licudata -L/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib -lm -ldl
Using vendored GLPK
x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/include" -DNDEBUG -DUSING_R -I. -Ivendor -Ivendor/cigraph/src -Ivendor/cigraph/include -Ivendor/cigraph/vendor -Ivendor/io/parsers  -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include/libxml2 -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -DHAVE_LIBXML -Ivendor/cigraph/vendor/glpk -Ivendor/cigraph/vendor/glpk/env -Ivendor/cigraph/vendor/glpk/minisat -Ivendor/cigraph/v

** libs
using C compiler: ‘x86_64-conda-linux-gnu-cc (Anaconda gcc) 11.2.0’
using C++ compiler: ‘x86_64-conda-linux-gnu-c++ (Anaconda gcc) 11.2.0’


x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/include" -DNDEBUG -DUSING_R -I. -Ivendor -Ivendor/cigraph/src -Ivendor/cigraph/include -Ivendor/cigraph/vendor -Ivendor/io/parsers  -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include/libxml2 -I/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/include -DHAVE_LIBXML -Ivendor/cigraph/vendor/glpk -Ivendor/cigraph/vendor/glpk/env -Ivendor/cigraph/vendor/glpk/minisat -Ivendor/cigraph/vendor/glpk/misc -Ivendor/cigraph/vendor/glpk/draft -Ivendor/cigraph/vendor/glpk/npp -Ivendor/cigraph/vendor/glpk/api -Ivendor/cigraph/vendor/glpk/mpl -Ivendor/cigraph/vendor/glpk/bflib -Ivendor/cigraph/vendor/glpk/amd -Ivendor/cigraph/vendor/glpk/simplex -Ivendor/cigraph/vendor/glpk/colamd -DNDEBUG -DNTIMER -DNPRINT -DIGRAPH_THREAD_LOCAL= -DPRPACK_IGRAPH_SUPPORT -DHAVE_GFORTRAN=1 -D_GNU_SOURCE=1 -I'/mnt/DATA/home/ethung/miniconda3/envs/scRNAseq/lib/R/library/cpp11/include' -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem

In [ ]:
%%R -i data_mat -o doublet_score -o doublet_class

set.seed(123)
sce = scDblFinder(
    SingleCellExperiment(
        list(counts=data_mat),
    ) 
)
doublet_score = sce$scDblFinder.score
doublet_class = sce$scDblFinder.class

#### Option 2: Scrublet
See: https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#doublet-detection

In [ ]:
sc.pp.scrublet(Data, batch_key="Sample")

#### Option 3: DoubletDetection
See: https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html#doublet-detection

In [ ]:
from doubletdetection import BoostClassifier

def Doublet_Filter(data):
    
    clf = BoostClassifier(
        n_iters=10,
        clustering_algorithm="leiden",
        standard_scaling=True,
        pseudocount=0.1,
        n_jobs=5)

    doublets = clf.fit(data.X).predict(p_thresh=1e-3, voter_thresh=0.5)
    doublet_score = clf.doublet_score()

    data.obs["doublet"] = doublets
    data.obs["doublet_score"] = doublet_score

    data.uns['doublets_removed'] = data.obs.doublet.sum()
    data = data[data.obs.doublet == 0]
    return data

data = [QC_Filter(dat) for dat in data]
for dat in data:
    print(len(dat), dat.uns['doublets_removed'])

In [ ]:
with open('paper_raw-processed.pickle', 'wb') as handle:
    pickle.dump(data, handle)

# Integration

In [ ]:
with open('paper_raw-processed.pickle', 'rb') as handle:
    data = pickle.load(handle)